# CS 228 experiment: keep E2's robustness, recover clean accuracy (hands-off)

**Inputs (right panel -> Add Input):** dataset `celeba-spoof-for-face-antispoofing` (Datasets tab) AND your notebook
`cs228-data` (Notebooks tab). Session options: GPU (T4 x2 or P100), Internet On.
**Save Version -> Save & Run All (Commit).** About 2 hours. Results: Versions -> newest -> Output (`bundle4.zip`) + log.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib
!git log --oneline -1

In [ ]:
# 2. Dataset location (attach it first: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing)
import glob, os
hits = []
for depth in range(1, 4):   # shallow search only: never walk the 750k-file dataset
    hits += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "celeba-spoof-for-face-antispoofing")
print("found:", hits)
assert hits, "Dataset not attached: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing"
SRC = hits[0]
print("dataset at:", SRC)

In [ ]:
# 3. Data: reuse the prebuilt data_pad2 from the attached cs228-data notebook, else build it
found = []
for depth in range(1, 6):
    found += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "data_pad2")
print("prebuilt data:", found)
if found:
    !ln -sfn {found[0]} ./data_pad2
else:
    !python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad2 --per_class 3000 --test_per_class 600 --max_per_subject 8
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad2/$s/$c | wc -l); done; done

In [ ]:
# 4. Baseline (no defense)
!python -m training.train_baseline --data_root ./data_pad2 --epochs 5 --seed 0 --out checkpoints/baseline

In [ ]:
# 5a. E2 from scratch (2/255), the reference defense
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --eps 0.00784 --out checkpoints/eps2

In [ ]:
# 5b. E2 fine-tuned FROM THE TRAINED BASELINE (keeps its screen-texture cues, then hardens them)
!python -m training.train_hardened --data_root ./data_pad2 --epochs 6 --lr 2e-5 --seed 0 --eps 0.00784 --init checkpoints/baseline/seed0_best.pt --out checkpoints/eps2_ft

In [ ]:
# 5c. E2 with a lighter adversarial weight (attacked images count 0.5x)
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --eps 0.00784 --adv_weight 0.5 --out checkpoints/eps2_light

In [ ]:
# 6. Test-set evaluation: sweep, attack success on fakes caught without attack, worst case over all budgets
B = '--baseline_glob "checkpoints/baseline/seed*_best.pt"'
for v in ["eps2", "eps2_ft", "eps2_light"]:
    !python -m evaluation.evaluate --data_root ./data_pad2 --sweep 1 2 4 8 16 {B} --hardened_glob "checkpoints/{v}/seed*_best.pt" --out results/eval_{v}.json
!python -m evaluation.make_report --run "E2 from scratch=results/eval_eps2.json" --run "E2 fine-tuned from baseline=results/eval_eps2_ft.json" --run "E2 light (0.5x)=results/eval_eps2_light.json" --out results/report4
print(open("results/report4.md").read())

In [ ]:
# 7. Which kinds of fakes each model accepts (clean images)
for ck in ["baseline"] + ["eps2", "eps2_ft", "eps2_light"]:
    !python -m evaluation.by_spoof_type --src {SRC} --data_root ./data_pad2 --ckpt checkpoints/{ck}/seed0_best.pt

In [ ]:
# 8. Pack the results
!zip -qr bundle4.zip results checkpoints/baseline/seed0_best.pt checkpoints/eps2/seed0_best.pt checkpoints/eps2_ft/seed0_best.pt checkpoints/eps2_light/seed0_best.pt
!ls -lh bundle4.zip